In [1]:
import os
# Должно быть установлено ДО импорта numpy/scipy, иначе не повлияет на OpenBLAS.
# implicit ALS использует собственный OpenMP-пул для параллелизма;
# BLAS должен быть однопоточным, чтобы не было oversubscription (24*24 потоков).
# OMP_NUM_THREADS намеренно НЕ трогаем — иначе убьём параллелизм implicit.
os.environ["OPENBLAS_NUM_THREADS"] = "1"

In [2]:
import polars as pl
import numpy as np

import src.dataset as dts
from src.modeling.ials import run_ials_experiment

2026-03-09 15:32:26.448 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: D:\HSE_repos\dreamteam-recsys


In [3]:
EVENTS_DIR = "../data/raw/dataset/small/marketplace/events"

events_df = (
    pl.scan_parquet(f"{EVENTS_DIR}/*.pq", include_file_paths="path")
    .sort("day")
)

In [4]:
# Таргет: inverse frequency weighting (аналогично SVD-бейзлайну)
actions_count = (
    events_df.group_by("action_type")
    .agg(pl.len())
    .collect(engine="streaming")
    .with_columns(
        (pl.col("len").sum() / pl.col("len")).alias("target"),
        (pl.col("len").sum() / pl.col("len")).log1p().alias("log_target"),
        (pl.col("len").sum() / pl.col("len")).sqrt().alias("sqrt_target"),
    )
    .drop("len")
)
actions_count

action_type,target,log_target,sqrt_target
str,f64,f64,f64
"""click""",34.582149,3.571844,5.880659
"""clickout""",268.714913,5.597366,16.392526
"""view""",1.034082,0.710044,1.016898
"""like""",3121.341812,8.046339,55.86897


In [5]:
events_df = events_df.join(actions_count.lazy(), on="action_type")

In [6]:
train, test = dts.global_temporal_split(events_df, 0.2)
print(f"Train schema: {train.collect_schema()}")
print(f"Test schema:  {test.collect_schema()}")

Train schema: Schema({'timestamp': Duration(time_unit='us'), 'user_id': UInt64, 'item_id': String, 'subdomain': String, 'action_type': String, 'os': String, 'day': Int32, 'path': String, 'target': Float64, 'log_target': Float64, 'sqrt_target': Float64})
Test schema:  Schema({'timestamp': Duration(time_unit='us'), 'user_id': UInt64, 'item_id': String, 'subdomain': String, 'action_type': String, 'os': String, 'day': Int32, 'path': String, 'target': Float64, 'log_target': Float64, 'sqrt_target': Float64})


In [7]:
# (target_col, factors, top_k, regularization, alpha, iterations)
experiment_configs = [
    # Базовое сравнение таргетов (аналогично SVD: factors=20, top_k=15)
    ("log_target", 20, 15, 0.01, 40.0, 30),
]

results = []
for t_col, factors, k, reg, alpha, iters in experiment_configs:
    res = run_ials_experiment(
        train, test,
        target_col=t_col,
        factors=factors,
        regularization=reg,
        alpha=alpha,
        iterations=iters,
        top_k=k,
    )
    results.append(res)

ials_results_df = pl.DataFrame(results)
ials_results_df

iALS: target=log_target, factors=20, reg=0.01, alpha=40.0, iter=30, top_k=15
Подготовка данных...
2026-03-09 02:07:00.682 | INFO     | src.modeling.ials:prepare_data:90 - Уникальных пар (user, item) после фильтрации: 65553749
2026-03-09 02:07:11.366 | INFO     | src.modeling.ials:prepare_data:117 - CSR-матрица: 1224053 users x 498229 items
Обучение модели...
2026-03-09 02:07:11.422 | INFO     | src.modeling.ials:fit:137 - Обучение iALS на CPU


  0%|          | 0/30 [00:00<?, ?it/s]

2026-03-09 02:10:29.225 | INFO     | src.modeling.ials:fit:152 - Модель iALS обучена
Обучение модели завершено
Пользователей в тесте после фильтрации: 633826
Сбор метрик...


target,factors,regularization,alpha,iterations,top_k,ndcg,precision,recall,rmse,mae
str,i64,f64,f64,i64,i64,f64,f64,f64,f64,f64
"""log_target""",20,0.01,40.0,30,15,0.12144,0.026098,0.026075,0.852188,0.608696


## Гипероптимизация (Optuna)

Обучать iALS на 1.2M пользователей при каждом trial дорого. Вместо этого:
1. Берём **40% пользователей** случайной выборкой (≈646k) — сохраняем тот же temporal split.
2. Запускаем Optuna на 25 прогонов, целевая метрика — **NDCG@15**.
3. Лучшую конфигурацию переобучаем на **полном датасете** для финального сравнения с SVD.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SUBSET_FRACTION = 0.4
OPTUNA_TRIALS = 25
TOP_K = 15
RANDOM_SEED = 42

# --- Сэмплируем подмножество пользователей ---
all_user_ids = (
    events_df.select("user_id").unique().collect(engine="streaming")["user_id"].to_list()
)
rng = np.random.default_rng(RANDOM_SEED)
subset_users = set(
    rng.choice(all_user_ids, size=int(len(all_user_ids) * SUBSET_FRACTION), replace=False).tolist()
)
print(f"Всего пользователей: {len(all_user_ids):,}")
print(f"Пользователей в подмножестве ({SUBSET_FRACTION*100:.0f}%): {len(subset_users):,}")

events_sub = events_df.filter(pl.col("user_id").is_in(subset_users))
train_sub, test_sub = dts.global_temporal_split(events_sub, 0.2)
print("Подмножество готово")

Всего пользователей: 1,616,763
Пользователей в подмножестве (40%): 646,705
Подмножество готово


In [8]:
from src.modeling.ials import run_ials_experiment

def optuna_objective(trial: optuna.Trial) -> float:
    target_col = trial.suggest_categorical("target_col", ["log_target", "sqrt_target"])
    factors    = trial.suggest_categorical("factors", [16, 32, 64, 128])
    reg        = trial.suggest_float("regularization", 1e-3, 1.0, log=True)
    alpha      = trial.suggest_float("alpha", 5.0, 200.0, log=True)
    iterations = trial.suggest_int("iterations", 10, 30)

    result = run_ials_experiment(
        train_sub, test_sub,
        target_col=target_col,
        factors=factors,
        regularization=reg,
        alpha=alpha,
        iterations=iterations,
        top_k=TOP_K,
        min_interactions=3,  # меньше из-за меньшего датасета
    )
    ndcg = result["ndcg"]
    trial.set_user_attr("precision", result["precision"])
    trial.set_user_attr("recall",    result["recall"])
    print(
        f"  Trial {trial.number}: target={target_col}, factors={factors}, "
        f"reg={reg:.4f}, alpha={alpha:.1f}, iter={iterations} → NDCG@{TOP_K}={ndcg:.6f}"
    )
    return ndcg

sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)

print(f"Запуск Optuna: {OPTUNA_TRIALS} trials на {SUBSET_FRACTION*100:.0f}% датасета...")
study.optimize(optuna_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)

best = study.best_trial
print(f"\nЛучший trial #{best.number}:")
print(f"  NDCG@{TOP_K} = {best.value:.6f}")
for k, v in best.params.items():
    print(f"  {k} = {v}")

Запуск Optuna: 25 trials на 40% датасета...
iALS: target=sqrt_target, factors=16, reg=0.0014936568554617625, alpha=122.07764786954151, iter=22, top_k=15
Подготовка данных...
2026-03-09 15:33:32.709 | INFO     | src.modeling.ials:prepare_data:90 - Уникальных пар (user, item) после фильтрации: 26360671
2026-03-09 15:33:35.649 | INFO     | src.modeling.ials:prepare_data:117 - CSR-матрица: 528139 users x 410599 items
Обучение модели...
2026-03-09 15:33:35.656 | INFO     | src.modeling.ials:fit:137 - Обучение iALS на CPU


  0%|          | 0/22 [00:00<?, ?it/s]

2026-03-09 15:34:06.914 | INFO     | src.modeling.ials:fit:152 - Модель iALS обучена
Обучение модели завершено
Пользователей в тесте после фильтрации: 263763
Сбор метрик...
  Trial 0: target=sqrt_target, factors=16, reg=0.0015, alpha=122.1, iter=22 → NDCG@15=0.091519
iALS: target=log_target, factors=16, reg=0.0035498788321965025, alpha=15.359756451337134, iter=21, top_k=15
Подготовка данных...
2026-03-09 15:37:56.322 | INFO     | src.modeling.ials:prepare_data:90 - Уникальных пар (user, item) после фильтрации: 26360671
2026-03-09 15:38:00.635 | INFO     | src.modeling.ials:prepare_data:117 - CSR-матрица: 528139 users x 410599 items
Обучение модели...
2026-03-09 15:38:00.651 | INFO     | src.modeling.ials:fit:137 - Обучение iALS на CPU


  0%|          | 0/21 [00:00<?, ?it/s]

2026-03-09 15:38:45.587 | INFO     | src.modeling.ials:fit:152 - Модель iALS обучена
Обучение модели завершено
Пользователей в тесте после фильтрации: 263763
Сбор метрик...
  Trial 1: target=log_target, factors=16, reg=0.0035, alpha=15.4, iter=21 → NDCG@15=0.134255
iALS: target=log_target, factors=16, reg=0.023345864076016236, alpha=90.54594344647792, iter=14, top_k=15
Подготовка данных...
2026-03-09 15:42:40.696 | INFO     | src.modeling.ials:prepare_data:90 - Уникальных пар (user, item) после фильтрации: 26360671
2026-03-09 15:42:44.989 | INFO     | src.modeling.ials:prepare_data:117 - CSR-матрица: 528139 users x 410599 items
Обучение модели...
2026-03-09 15:42:45.005 | INFO     | src.modeling.ials:fit:137 - Обучение iALS на CPU


...

2026-03-09 17:23:34.415 | INFO     | src.modeling.ials:fit:152 - Модель iALS обучена
Обучение модели завершено
Пользователей в тесте после фильтрации: 263763
Сбор метрик...
  Trial 22: target=log_target, factors=16, reg=0.1069, alpha=8.3, iter=15 → NDCG@15=0.141719
iALS: target=log_target, factors=16, reg=0.11164838757266801, alpha=7.874580908973367, iter=16, top_k=15
Подготовка данных...
2026-03-09 17:26:47.322 | INFO     | src.modeling.ials:prepare_data:90 - Уникальных пар (user, item) после фильтрации: 26360671
2026-03-09 17:26:51.300 | INFO     | src.modeling.ials:prepare_data:117 - CSR-матрица: 528139 users x 410599 items
Обучение модели...
2026-03-09 17:26:51.315 | INFO     | src.modeling.ials:fit:137 - Обучение iALS на CPU


  0%|          | 0/16 [00:00<?, ?it/s]

2026-03-09 17:27:15.190 | INFO     | src.modeling.ials:fit:152 - Модель iALS обучена
Обучение модели завершено
Пользователей в тесте после фильтрации: 263763
Сбор метрик...
  Trial 23: target=log_target, factors=16, reg=0.1116, alpha=7.9, iter=16 → NDCG@15=0.138508
iALS: target=log_target, factors=16, reg=0.37369186371892243, alpha=44.96624334977962, iter=12, top_k=15
Подготовка данных...
2026-03-09 17:30:12.437 | INFO     | src.modeling.ials:prepare_data:90 - Уникальных пар (user, item) после фильтрации: 26360671
2026-03-09 17:30:15.470 | INFO     | src.modeling.ials:prepare_data:117 - CSR-матрица: 528139 users x 410599 items
Обучение модели...
2026-03-09 17:30:15.483 | INFO     | src.modeling.ials:fit:137 - Обучение iALS на CPU


  0%|          | 0/12 [00:00<?, ?it/s]

2026-03-09 17:30:34.282 | INFO     | src.modeling.ials:fit:152 - Модель iALS обучена
Обучение модели завершено
Пользователей в тесте после фильтрации: 263763
Сбор метрик...
  Trial 24: target=log_target, factors=16, reg=0.3737, alpha=45.0, iter=12 → NDCG@15=0.120079

Лучший trial #21:
  NDCG@15 = 0.142534
  target_col = log_target
  factors = 16
  regularization = 0.13298157149772175
  alpha = 9.042146805118975
  iterations = 18


In [9]:
# Сводная таблица всех trials
trials_df = pl.DataFrame([
    {
        "trial":          t.number,
        "ndcg":           t.value,
        "precision":      t.user_attrs.get("precision"),
        "recall":         t.user_attrs.get("recall"),
        "target_col":     t.params["target_col"],
        "factors":        t.params["factors"],
        "regularization": t.params["regularization"],
        "alpha":          t.params["alpha"],
        "iterations":     t.params["iterations"],
    }
    for t in study.trials
    if t.state == optuna.trial.TrialState.COMPLETE
]).sort("ndcg", descending=True)

trials_df

trial,ndcg,precision,recall,target_col,factors,regularization,alpha,iterations
i64,f64,f64,f64,str,i64,f64,f64,i64
21,0.142534,0.029941,0.029993,"""log_target""",16,0.132982,9.042147,18
22,0.141719,0.02944,0.029716,"""log_target""",16,0.106914,8.274813,15
19,0.139855,0.029291,0.029139,"""log_target""",16,0.154682,9.271621,17
23,0.138508,0.028992,0.028875,"""log_target""",16,0.111648,7.874581,16
16,0.138338,0.028966,0.028981,"""log_target""",16,0.056193,12.188647,18
…,…,…,…,…,…,…,…,…
7,0.091551,0.02031,0.021978,"""log_target""",64,0.9134,86.328154,14
0,0.091519,0.019442,0.021763,"""sqrt_target""",16,0.001494,122.077648,22
9,0.082572,0.018541,0.020861,"""log_target""",128,0.081785,131.928323,19


## Финальная оценка лучшего iALS на полном датасете

Переобучаем модель с найденными гиперпараметрами на **полном** train/test split (тот же, что использовался для SVD) — только так сравнение корректно.

In [10]:
best_params = best.params
print("Лучшие гиперпараметры (из hyperopt на subset):")
for k, v in best_params.items():
    print(f"  {k}: {v}")

print("\nОбучение на полном датасете...")
best_ials_result = run_ials_experiment(
    train, test,
    target_col=best_params["target_col"],
    factors=best_params["factors"],
    regularization=best_params["regularization"],
    alpha=best_params["alpha"],
    iterations=best_params["iterations"],
    top_k=TOP_K,
)

print("\nРезультат лучшего iALS (full data):")
for metric in ("ndcg", "precision", "recall", "rmse", "mae"):
    print(f"  {metric}: {best_ials_result[metric]:.6f}")

Лучшие гиперпараметры (из hyperopt на subset):
  target_col: log_target
  factors: 16
  regularization: 0.13298157149772175
  alpha: 9.042146805118975
  iterations: 18

Обучение на полном датасете...
iALS: target=log_target, factors=16, reg=0.13298157149772175, alpha=9.042146805118975, iter=18, top_k=15
Подготовка данных...
2026-03-09 17:33:45.143 | INFO     | src.modeling.ials:prepare_data:90 - Уникальных пар (user, item) после фильтрации: 65553749
2026-03-09 17:33:53.643 | INFO     | src.modeling.ials:prepare_data:117 - CSR-матрица: 1224053 users x 498229 items
Обучение модели...
2026-03-09 17:33:53.699 | INFO     | src.modeling.ials:fit:137 - Обучение iALS на CPU


  0%|          | 0/18 [00:00<?, ?it/s]

2026-03-09 17:34:59.756 | INFO     | src.modeling.ials:fit:152 - Модель iALS обучена
Обучение модели завершено
Пользователей в тесте после фильтрации: 633826
Сбор метрик...

Результат лучшего iALS (full data):
  ndcg: 0.143952
  precision: 0.030002
  recall: 0.029478
  rmse: 0.890737
  mae: 0.664983


## Сравнение с бейзлайном (SVD)

Условия одинаковые для обоих алгоритмов:
- **Данные**: один и тот же датасет, одинаковый global temporal split 80/20
- **Метрика ранжирования**: NDCG@15, Precision@15, Recall@15
- **SVD**: лучший результат из `svd_experiments_unified.ipynb` — `log_target`, factors=20
- **iALS**: конфигурация, найденная Optuna на 40%-подмножестве, оцененная на полных данных

In [ ]:
svd_best = {
    "model":          "SVD (best)",
    "target":         "log_target",
    "factors":        20,
    "top_k":          15,
    "ndcg":           0.135736,
    "precision":      0.026741,
    "recall":         0.026168,
    "rmse":           3.111453,
    "mae":            0.994588,
    "hyperopt":       "grid (ручной)",
}

ials_best = {
    "model":          "iALS (best, Optuna)",
    "target":         best_ials_result["target"],
    "factors":        best_ials_result["factors"],
    "top_k":          best_ials_result["top_k"],
    "ndcg":           best_ials_result["ndcg"],
    "precision":      best_ials_result["precision"],
    "recall":         best_ials_result["recall"],
    "rmse":           best_ials_result["rmse"],
    "mae":            best_ials_result["mae"],
    "hyperopt":       f"Optuna TPE, {OPTUNA_TRIALS} trials, {SUBSET_FRACTION*100:.0f}% subset",
}

final_comparison = pl.DataFrame([svd_best, ials_best])

# Добавим delta (iALS vs SVD) по ключевым метрикам
ndcg_delta    = ials_best["ndcg"]      - svd_best["ndcg"]
prec_delta    = ials_best["precision"] - svd_best["precision"]
recall_delta  = ials_best["recall"]    - svd_best["recall"]

print(f"NDCG@15:    SVD={svd_best['ndcg']:.6f}  iALS={ials_best['ndcg']:.6f}  Δ={ndcg_delta:+.6f} ({ndcg_delta/svd_best['ndcg']*100:+.1f}%)")
print(f"Precision:  SVD={svd_best['precision']:.6f}  iALS={ials_best['precision']:.6f}  Δ={prec_delta:+.6f}")
print(f"Recall:     SVD={svd_best['recall']:.6f}  iALS={ials_best['recall']:.6f}  Δ={recall_delta:+.6f}")

final_comparison

NDCG@15:    SVD=0.135736  iALS=0.143952  Δ=+0.008216 (+6.1%)
Precision:  SVD=0.026741  iALS=0.030002  Δ=+0.003261
Recall:     SVD=0.026168  iALS=0.029478  Δ=+0.003310


model,target,factors,top_k,ndcg,precision,recall,rmse,mae,hyperopt
str,str,i64,i64,f64,f64,f64,f64,f64,str
"""SVD (best)""","""log_target""",20,15,0.135736,0.026741,0.026168,3.111453,0.994588,"""grid (ручной)"""
"""iALS (best, Optuna)""","""log_target""",16,15,0.143952,0.030002,0.029478,0.890737,0.664983,"""Optuna TPE, 25 trials, 40% sub…"


## Выводы

- Лучшая конфигурация iALS, найденная Optuna: `target_col=log_target, factors=16, reg=0.133, alpha=9, iter=18`
- По метрике NDCG@15 iALS лучше SVD на 6.1%
- Было замечено, что повышение кол-ва факторов для iALS приводит к ухудшению качества - модель запоминает больше "ненужной" информации, из-за чего дает более шумные рекомендации.
- Преимущество iALS перед SVD: учитывает implicit feedback напрямую через confidence weighting, лучше обрабатывает разреженность